# AI PCB Inspector — full-board training (Colab only)

This notebook owns **all** development, training and export. Your PC/Pi only downloads a run bundle and runs `test_board.py` on a whole-board image.

What changed from the old pilot:
- The model accepts a **full board image**, finds joints itself, and reports boxes + labels. No caller-supplied crops.
- SolDef `no_good` (component misplacement) is mapped to `other`, **never** to `solder_bridge`. Unreviewed mappings cannot silently become ground truth.
- If there is no reviewed bridge source, the `solder_bridge` class is dropped from `labels.txt` so the model physically cannot call a bridge.
- One command publishes a TFLite bundle, a model card, checksums, handoff state, and a GitHub push.

**Secrets to add in Colab Secrets (left sidebar 🔑), never in a cell or chat:** `KAGGLE_USERNAME`, `KAGGLE_KEY`, `ROBOFLOW_API_KEY` (optional), `GH_TOKEN` (PAT with write access).

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive, userdata

drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/AI-PCB-Inspector/repo')
assert REPO.is_dir(), 'Repo folder missing in Drive'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Repo:', REPO)
print('Colab-only workflow: no local sync step.')

In [ ]:
!pip -q install -U ultralytics roboflow kaggle tensorflow
import torch, tensorflow as tf, ultralytics
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('tensorflow', tf.__version__, 'ultralytics', ultralytics.__version__)

In [ ]:
# Pull secrets from Colab Secrets into the environment for the CLI.
# Nothing is printed. Do NOT paste keys into this notebook.
for name in ('KAGGLE_USERNAME', 'KAGGLE_KEY', 'ROBOFLOW_API_KEY', 'GH_TOKEN', 'GITHUB_TOKEN'):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value
        print(f'{name}: set')
    else:
        print(f'{name}: missing (optional except GH_TOKEN)')

## 1. Download + visual label audit

Contact sheets appear under `runs/audit/<source>/<class>_contact.jpg`. Open them and decide which mappings are defensible. Edit `training/configs/fullboard_config.json`: set `"reviewed": true` only for mappings you can defend with an example image. Leave `solder_bridge` unmapped unless you see actual pad-to-pad bridging.

In [ ]:
CMD = 'python training/fullboard_colab.py'
CFG = 'training/configs/fullboard_config.json'
!{CMD} download --config {CFG}
!{CMD} audit --config {CFG} --previews 8
from IPython.display import display, Image as DisplayImage
audit_dir = Path('/content/drive/MyDrive/AI-PCB-Inspector/runs/audit')
for sheet in sorted(audit_dir.rglob('*_contact.jpg')):
    print(sheet.relative_to(audit_dir))
    display(DisplayImage(filename=str(sheet)))

## 2. Train + export + publish

`ALLOW_PROVISIONAL = True` lets the pipeline run before every mapping is reviewed. The run is still stamped with a `label_semantics` warning and the model card says which mappings are provisional. Set it to `False` once the audit is signed off.

In [ ]:
WORK = '/content/pcb_fullboard_run'
ALLOW_PROVISIONAL = True
DEVICE = 'auto'
flag = ' --allow-provisional' if ALLOW_PROVISIONAL else ''
!{CMD} prepare --config {CFG} --output {WORK}{flag}
!{CMD} train --work {WORK} --device {DEVICE}
!{CMD} export --work {WORK}

In [ ]:
# Publish the bundle to Drive handoff state and push the cleaned repo to GitHub.
import glob
run_dirs = sorted(glob.glob('/content/drive/MyDrive/AI-PCB-Inspector/runs/*/model_card.json'))
latest = max(run_dirs, key=os.path.getmtime)
latest_dir = str(Path(latest).parent)
!{CMD} publish --run-dir '{latest_dir}'
!{CMD} push --message 'Colab: full-board inspector overhaul'

In [ ]:
# Verify the stable export bundle before downloading it locally.
import hashlib
current = Path('/content/drive/MyDrive/AI-PCB-Inspector/exports/current')
manifest = (current / 'manifest.sha256').read_text().splitlines()
for line in manifest:
    digest, name = line.split('  ', 1)
    actual = hashlib.sha256((current / name).read_bytes()).hexdigest()
    assert actual == digest, f'CHECKSUM MISMATCH: {name}'
print('Bundle OK:', current)
print('Files:', sorted(p.name for p in current.iterdir()))
print(json.dumps(json.loads((current / 'model_card.json').read_text())['classes'], indent=2))